# Cleaning settlement data for Belarus

This notebook prepares settlement points for the healthcare-access analysis.

**Source:** [HOTOSM Belarus Populated Places](https://data.humdata.org/dataset/hotosm_blr_populated_places)

## 1. Load the data

The source GeoPackage contains populated places from OpenStreetMap.

In [2]:
import geopandas as gpd

settlements = gpd.read_file("map_data/populated_places.gpkg")

## 2. Inspect the source data

This check shows the available columns, data types, number of records, and coordinate reference system.

In [3]:
settlements.info()

print(f"Rows loaded: {len(settlements):,}")
print(f"CRS: {settlements.crs}")

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 127723 entries, 0 to 127722
Data columns (total 22 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   id          127723 non-null  str     
 1   name        45954 non-null   str     
 2   name_en     4413 non-null    str     
 3   name_be     45655 non-null   str     
 4   name_ru     45676 non-null   str     
 5   place       45228 non-null   str     
 6   landuse     82781 non-null   str     
 7   population  10552 non-null   str     
 8   is_in       0 non-null       object  
 9   source      0 non-null       object  
 10  adm0_pcode  127723 non-null  str     
 11  adm0_name   127723 non-null  str     
 12  adm1_pcode  127451 non-null  str     
 13  adm1_name   127451 non-null  str     
 14  adm2_pcode  127451 non-null  str     
 15  adm2_name   127451 non-null  str     
 16  adm3_pcode  0 non-null       object  
 17  adm3_name   0 non-null       object  
 18  adm4_pcode  0 no

## 3. Keep settlement points with Russian names

OpenStreetMap node IDs are used to remove polygon and line features. Records without a Russian settlement name are also removed.

In [4]:
# Keep point features represented by OpenStreetMap nodes
settlements = settlements[
    settlements["id"].str.startswith("node/")
].copy()

In [5]:
settlements = settlements[
    settlements["name_ru"].notna()
].copy()

## 4. Keep relevant settlement types

The analysis includes cities, towns, villages, hamlets, and isolated dwellings.

In [6]:
valid_places = [
    "city",
    "town",
    "village",
    "hamlet",
    "isolated_dwelling"
]

settlements = settlements[
    settlements["place"].isin(valid_places)
].copy()

## 5. Create a cleaned settlement key

Russian names are stripped of surrounding spaces, and `ё` is standardized to `е` for later matching.

In [7]:
settlements["settlement_key"] = (
    settlements["name_ru"]
    .str.strip()
    .str.replace("ё", "е", regex=False)
    .str.replace("Ё", "Е", regex=False)
)

## 6. Keep the required columns

Only identifiers, settlement names, administrative fields, geometry, and the cleaned matching key are retained.

In [8]:
settlements = settlements[
    [
        "id",
        "name",
        "name_ru",
        "name_be",
        "name_en",
        "name_latin",
        "place",
        "adm0_name",
        "adm1_pcode",
        "adm1_name",
        "adm2_pcode",
        "adm2_name",
        "geometry",
        "settlement_key",
    ]
].copy()

In [9]:
print(f"Settlements retained: {len(settlements):,}")
print(settlements["place"].value_counts())

Settlements retained: 22,477
place
hamlet               19142
village               2468
isolated_dwelling      662
town                   190
city                    15
Name: count, dtype: int64


## 7. Export the cleaned data

The GeoPackage preserves the spatial data. The CSV provides a tabular copy for later joins and checks.

In [13]:
settlements.to_csv(
    "output_data/csv_final_settlements.csv",
    index=False
)

In [12]:
settlements.to_csv(
    "output_data/csv_final_settlements.csv",
    index=False
)